# CELL 1: Setup and Imports

In [ ]:
# # CELL 0: Install Required Dependencies (Run First!)
# # ============================================================================
# # Installing from requirements.txt with pygraphviz fix for Kaggle

# import sys
# import subprocess

# print("📦 Installing required dependencies for HW2VEC...")
# print("="*60)

# # Read requirements.txt from hw2vec
# requirements_path = "/kaggle/input/ht-detection-gnn/hw2vec-master/requirements.txt"

# print(f"\n📄 Reading requirements from: {requirements_path}")
# with open(requirements_path, 'r') as f:
#     requirements = [line.strip() for line in f if line.strip() and not line.startswith('#')]

# print(f"   Found {len(requirements)} packages to install")
# print(f"   Packages: {', '.join(requirements)}")

# # Install each package, handling pygraphviz specially
# print("\n" + "="*60)
# for idx, package in enumerate(requirements, 1):
#     print(f"\n{idx}️⃣ Installing {package}...")
    
#     # Skip pygraphviz - it requires system libraries that Kaggle doesn't have
#     if 'pygraphviz' in package.lower():
#         print(f"   ⚠️  SKIPPING pygraphviz (requires graphviz system library)")
#         print(f"   ℹ️  This is OK - pygraphviz is OPTIONAL for visualization only")
#         print(f"   ℹ️  We use pydot instead (already available in Kaggle)")
#         continue
    
#     # Install the package
#     try:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
#         print(f"   ✅ {package} installed successfully")
#     except subprocess.CalledProcessError as e:
#         print(f"   ❌ Failed to install {package}: {e}")
#         print(f"   ⚠️  Continuing anyway...")

# # Install torch_geometric separately (not in requirements.txt)
# print("\n" + "="*60)
# print("\n🔧 Installing torch_geometric (GNN library)...")
# try:
#     # Install torch_geometric dependencies
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
#                           "torch-scatter", "torch-sparse", "torch-cluster", "torch-spline-conv",
#                           "-f", "https://data.pyg.org/whl/torch-2.0.0+cu118.html"])
    
#     # Install torch_geometric
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
#     print("   ✅ torch_geometric installed successfully")
# except subprocess.CalledProcessError as e:
#     print(f"   ❌ torch_geometric installation failed: {e}")

# print("\n" + "="*60)
# print("✅ Installation complete!")
# print("\n📋 Summary:")
# print("   ✅ pyverilog: Installed (Verilog parser)")
# print("   ✅ torch_geometric: Installed (GNN library)")
# print("   ✅ pydot: Installed (graph visualization)")
# print("   ⚠️  pygraphviz: SKIPPED (not needed - optional)")
# print("\n💡 Note: pygraphviz error is EXPECTED and HARMLESS")
# print("   We don't need it for trojan detection!")
# print("="*60)

In [ ]:
# !apt-get update -qq
# !apt-get install -y graphviz graphviz-dev
# !pip install -q pygraphviz tqdm==4.66.4 huggingface_hub==0.25.1 matplotlib==3.5.3 torch-geometric==2.4.0


In [ ]:
# !pip install -U tqdm==4.66.4 huggingface_hub==0.25.1


In [ ]:
# !pip install matplotlib==3.6.3 tqdm==4.64.1 --force-reinstall
# !pip install pygraphviz==1.7 networkx==2.8.8 huggingface_hub==0.25.1
# !pip uninstall -y matplotlib tqdm huggingface_hub pygraphviz networkx
# !pip install matplotlib==3.5.3 tqdm==4.64.1 huggingface_hub==0.25.1 pygraphviz==1.7 networkx==2.8.8 scikit-learn==1.1.3 pandas==1.5.3 torch==2.0.1


In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path
import torch
import pickle as pkl
from datetime import datetime

# Add hw2vec to path
hw2vec_path = "/kaggle/input/ht-detection-gnn/hw2vec-master"
sys.path.append(hw2vec_path)

from hw2vec.hw2graph import HW2GRAPH, DataProcessor
from hw2vec.graph2vec.models import GRAPH2VEC
from hw2vec.config import Config
from torch_geometric.data import DataLoader

print("✅ All imports successful!")
print(f"📁 HW2VEC path: {hw2vec_path}")

# CELL 2: Configuration and Pre-trained Model Paths

Sets paths to DFG and AST pre-trained models
Verifies models exist
Configures device (GPU/CPU)



In [ ]:
# Pre-trained model paths
DFG_MODEL_CFG = f"{hw2vec_path}/assets/datasets/pretrained_DFG_TJ_RTL/model.cfg"
DFG_MODEL_PTH = f"{hw2vec_path}/assets/datasets/pretrained_DFG_TJ_RTL/model.pth"

AST_MODEL_CFG = f"{hw2vec_path}/assets/datasets/pretrained_AST_TJ_RTL/model.cfg"
AST_MODEL_PTH = f"{hw2vec_path}/assets/datasets/pretrained_AST_TJ_RTL/model.pth"

# Verify pre-trained models exist
assert os.path.exists(DFG_MODEL_CFG), f"DFG config not found: {DFG_MODEL_CFG}"
assert os.path.exists(DFG_MODEL_PTH), f"DFG weights not found: {DFG_MODEL_PTH}"
assert os.path.exists(AST_MODEL_CFG), f"AST config not found: {AST_MODEL_CFG}"
assert os.path.exists(AST_MODEL_PTH), f"AST weights not found: {AST_MODEL_PTH}"

print("✅ Pre-trained models verified:")
print(f"   DFG Model: {DFG_MODEL_PTH}")
print(f"   AST Model: {AST_MODEL_PTH}")

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {DEVICE}")

# CELL 3: File Preparation Class

Reorganizes your trojan files into HW2VEC structure
Creates TjFree/ and TjIn/ folders
Copies original + trojan designs

In [ ]:
class TrojanFilePreparation:
    """Prepares trojan files for HW2VEC processing"""
    
    def __init__(self, trojan_output_dir="/kaggle/working/trojan_output"):
        self.trojan_output_dir = Path(trojan_output_dir)
        self.hw2vec_dataset_dir = Path("/kaggle/working/hw2vec_dataset")
        
    def prepare_dataset(self):
        """Reorganize files into HW2VEC expected structure"""
        
        print("📦 Preparing dataset for HW2VEC...")
        
        # Create HW2VEC dataset structure
        tjfree_dir = self.hw2vec_dataset_dir / "TjFree" / "original"
        tjin_dir = self.hw2vec_dataset_dir / "TjIn"
        
        tjfree_dir.mkdir(parents=True, exist_ok=True)
        tjin_dir.mkdir(parents=True, exist_ok=True)
        
        # Copy original file (clean design)
        original_file = self.trojan_output_dir / "RISCV_ORIGINAL.sv"
        if original_file.exists():
            shutil.copy(original_file, tjfree_dir / "RISCV_ORIGINAL.v")
            print(f"   ✅ Copied clean design: RISCV_ORIGINAL.v")
        else:
            print(f"   ⚠️  Original file not found: {original_file}")
            return None
        
        # Copy trojan-infected files
        integrated_dir = self.trojan_output_dir / "complete_integrated_files"
        trojan_count = 0
        
        if integrated_dir.exists():
            for sv_file in integrated_dir.glob("*.sv"):
                # Create subfolder for each trojan design
                trojan_name = sv_file.stem  # e.g., "RISCV_with_HT001_HT2_Leakage"
                trojan_folder = tjin_dir / trojan_name
                trojan_folder.mkdir(exist_ok=True)
                
                # Copy and rename to .v extension
                shutil.copy(sv_file, trojan_folder / f"{trojan_name}.v")
                trojan_count += 1
        
        print(f"   ✅ Copied {trojan_count} trojan designs")
        print(f"   📁 Dataset ready at: {self.hw2vec_dataset_dir}")
        
        return {
            "dataset_path": str(self.hw2vec_dataset_dir),
            "tjfree_count": 1,
            "tjin_count": trojan_count,
            "total_designs": trojan_count + 1
        }

# Test file preparation
prep = TrojanFilePreparation()
dataset_info = prep.prepare_dataset()

if dataset_info:
    print(f"\n📊 Dataset Summary:")
    print(f"   Clean designs: {dataset_info['tjfree_count']}")
    print(f"   Trojan designs: {dataset_info['tjin_count']}")
    print(f"   Total designs: {dataset_info['total_designs']}")


# CELL 4: Graph Generation Helper

Generates DFG/AST graphs from Verilog files
Processes entire dataset

In [ ]:
class GraphGenerator:
    """Generate graphs from Verilog designs"""
    
    def __init__(self, graph_type="DFG"):
        """
        Args:
            graph_type: "DFG" or "AST"
        """
        self.graph_type = graph_type
        
        # Create minimal config for graph generation
        self.cfg = type('Config', (), {})()
        self.cfg.graph_type = graph_type
        self.cfg.device = DEVICE
        self.cfg.raw_dataset_path = Path("/kaggle/working/hw2vec_dataset")
        
        self.hw2graph = HW2GRAPH(self.cfg)
        self.data_proc = DataProcessor(self.cfg)
        
    def generate_graphs_from_dataset(self):
        """Generate graphs for all designs in dataset"""
        
        print(f"🔄 Generating {self.graph_type} graphs...")
        
        nx_graphs = []
        project_folders = self.hw2graph.find_hw_project_folders()
        
        print(f"   Found {len(project_folders)} design folders")
        
        for idx, hw_project_path in enumerate(project_folders, 1):
            try:
                print(f"   Processing {idx}/{len(project_folders)}: {hw_project_path.name}")
                hw_graph = self.hw2graph.code2graph(hw_project_path, profile=False)
                nx_graphs.append(hw_graph)
                self.data_proc.process(hw_graph)
            except Exception as e:
                print(f"   ⚠️  Error processing {hw_project_path.name}: {e}")
        
        print(f"   ✅ Generated {len(nx_graphs)} graphs")
        return self.data_proc.get_graphs()

# CELL 5: Trojan Detection Model (DFG)

Loads DFG pre-trained model
Provides test_single_design() function
Provides test_dataset() function
NO TRAINING - pure inference only

In [ ]:
class TrojanDetectorDFG:
    """Hardware Trojan Detector using DFG pre-trained model"""
    
    def __init__(self):
        self.graph_type = "DFG"
        
        # Create config
        self.cfg = type('Config', (), {})()
        self.cfg.graph_type = "DFG"
        self.cfg.device = DEVICE
        self.cfg.dropout = 0.5
        self.cfg.num_feature_dim = None  # Will be set after loading model
        
        # Initialize model
        self.model = GRAPH2VEC(self.cfg)
        self.load_pretrained_model()
        
    def load_pretrained_model(self):
        """Load pre-trained DFG model"""
        print(f"📥 Loading DFG pre-trained model...")
        
        try:
            self.model.load_model(DFG_MODEL_CFG, DFG_MODEL_PTH)
            self.model.to(DEVICE)
            self.model.eval()
            print(f"   ✅ DFG model loaded successfully")
        except Exception as e:
            print(f"   ❌ Error loading DFG model: {e}")
            raise
    
    def test_single_design(self, verilog_file):
        """
        Test a single Verilog design file
        
        Args:
            verilog_file: Path to .v or .sv file
            
        Returns:
            dict with prediction results
        """
        print(f"\n🧪 Testing single design with DFG model...")
        print(f"   File: {verilog_file}")
        
        # Create temporary dataset structure
        temp_dir = Path("/tmp/hw2vec_test")
        test_dir = temp_dir / "test_design"
        test_dir.mkdir(parents=True, exist_ok=True)
        
        # Copy file
        shutil.copy(verilog_file, test_dir / "design.v")
        
        # Generate graph
        cfg_temp = type('Config', (), {})()
        cfg_temp.graph_type = "DFG"
        cfg_temp.device = DEVICE
        cfg_temp.raw_dataset_path = temp_dir
        
        hw2graph = HW2GRAPH(cfg_temp)
        hw_graph = hw2graph.code2graph(test_dir, profile=False)
        
        data_proc = DataProcessor(cfg_temp)
        data_proc.process(hw_graph)
        graphs = data_proc.get_graphs()
        
        # Run inference
        loader = DataLoader(graphs, batch_size=1)
        data = next(iter(loader)).to(DEVICE)
        
        with torch.no_grad():
            graph_embed, _ = self.model.embed_graph(data.x, data.edge_index, data.batch)
            output = self.model.mlp(graph_embed)
            probabilities = torch.softmax(output, dim=1)
            prediction = output.max(1)[1].item()
        
        result = {
            "model": "DFG",
            "file": str(verilog_file),
            "prediction": "TROJAN" if prediction == 1 else "CLEAN",
            "confidence": probabilities[0][prediction].item(),
            "prob_clean": probabilities[0][0].item(),
            "prob_trojan": probabilities[0][1].item()
        }
        
        print(f"   🎯 Result: {result['prediction']} (confidence: {result['confidence']:.3f})")
        
        # Cleanup
        shutil.rmtree(temp_dir)
        
        return result
    
    def test_dataset(self, graph_data):
        """Test all designs in a graph dataset"""
        
        print(f"\n🧪 Testing dataset with DFG model...")
        
        results = []
        loader = DataLoader(graph_data, batch_size=1)
        
        with torch.no_grad():
            for data in loader:
                data = data.to(DEVICE)
                
                graph_embed, _ = self.model.embed_graph(data.x, data.edge_index, data.batch)
                output = self.model.mlp(graph_embed)
                probabilities = torch.softmax(output, dim=1)
                prediction = output.max(1)[1].item()
                
                result = {
                    "model": "DFG",
                    "design_name": data.hw_name[0],
                    "design_type": data.hw_type[0],
                    "prediction": "TROJAN" if prediction == 1 else "CLEAN",
                    "confidence": probabilities[0][prediction].item(),
                    "prob_clean": probabilities[0][0].item(),
                    "prob_trojan": probabilities[0][1].item()
                }
                
                results.append(result)
                
                status = "✓" if (result['design_type'] == "TjIn" and prediction == 1) or \
                               (result['design_type'] == "TjFree" and prediction == 0) else "✗"
                
                print(f"   {status} {result['design_name']}: {result['prediction']} ({result['confidence']:.3f})")
        
        return results


# Initialize DFG detector
dfg_detector = TrojanDetectorDFG()
print("✅ DFG Detector ready!")

# CELL 6: Trojan Detection Model (AST)

Loads AST pre-trained model
Same functions as DFG
NO TRAINING - pure inference only

In [ ]:
class TrojanDetectorAST:
    """Hardware Trojan Detector using AST pre-trained model"""
    
    def __init__(self):
        self.graph_type = "AST"
        
        # Create config
        self.cfg = type('Config', (), {})()
        self.cfg.graph_type = "AST"
        self.cfg.device = DEVICE
        self.cfg.dropout = 0.5
        self.cfg.num_feature_dim = None
        
        # Initialize model
        self.model = GRAPH2VEC(self.cfg)
        self.load_pretrained_model()
        
    def load_pretrained_model(self):
        """Load pre-trained AST model"""
        print(f"📥 Loading AST pre-trained model...")
        
        try:
            self.model.load_model(AST_MODEL_CFG, AST_MODEL_PTH)
            self.model.to(DEVICE)
            self.model.eval()
            print(f"   ✅ AST model loaded successfully")
        except Exception as e:
            print(f"   ❌ Error loading AST model: {e}")
            raise
    
    def test_single_design(self, verilog_file):
        """Test a single Verilog design file"""
        print(f"\n🧪 Testing single design with AST model...")
        print(f"   File: {verilog_file}")
        
        # Create temporary dataset structure
        temp_dir = Path("/tmp/hw2vec_test_ast")
        test_dir = temp_dir / "test_design"
        test_dir.mkdir(parents=True, exist_ok=True)
        
        # Copy file
        shutil.copy(verilog_file, test_dir / "design.v")
        
        # Generate graph
        cfg_temp = type('Config', (), {})()
        cfg_temp.graph_type = "AST"
        cfg_temp.device = DEVICE
        cfg_temp.raw_dataset_path = temp_dir
        
        hw2graph = HW2GRAPH(cfg_temp)
        hw_graph = hw2graph.code2graph(test_dir, profile=False)
        
        data_proc = DataProcessor(cfg_temp)
        data_proc.process(hw_graph)
        graphs = data_proc.get_graphs()
        
        # Run inference
        loader = DataLoader(graphs, batch_size=1)
        data = next(iter(loader)).to(DEVICE)
        
        with torch.no_grad():
            graph_embed, _ = self.model.embed_graph(data.x, data.edge_index, data.batch)
            output = self.model.mlp(graph_embed)
            probabilities = torch.softmax(output, dim=1)
            prediction = output.max(1)[1].item()
        
        result = {
            "model": "AST",
            "file": str(verilog_file),
            "prediction": "TROJAN" if prediction == 1 else "CLEAN",
            "confidence": probabilities[0][prediction].item(),
            "prob_clean": probabilities[0][0].item(),
            "prob_trojan": probabilities[0][1].item()
        }
        
        print(f"   🎯 Result: {result['prediction']} (confidence: {result['confidence']:.3f})")
        
        # Cleanup
        shutil.rmtree(temp_dir)
        
        return result
    
    def test_dataset(self, graph_data):
        """Test all designs in a graph dataset"""
        
        print(f"\n🧪 Testing dataset with AST model...")
        
        results = []
        loader = DataLoader(graph_data, batch_size=1)
        
        with torch.no_grad():
            for data in loader:
                data = data.to(DEVICE)
                
                graph_embed, _ = self.model.embed_graph(data.x, data.edge_index, data.batch)
                output = self.model.mlp(graph_embed)
                probabilities = torch.softmax(output, dim=1)
                prediction = output.max(1)[1].item()
                
                result = {
                    "model": "AST",
                    "design_name": data.hw_name[0],
                    "design_type": data.hw_type[0],
                    "prediction": "TROJAN" if prediction == 1 else "CLEAN",
                    "confidence": probabilities[0][prediction].item(),
                    "prob_clean": probabilities[0][0].item(),
                    "prob_trojan": probabilities[0][1].item()
                }
                
                results.append(result)
                
                status = "✓" if (result['design_type'] == "TjIn" and prediction == 1) or \
                               (result['design_type'] == "TjFree" and prediction == 0) else "✗"
                
                print(f"   {status} {result['design_name']}: {result['prediction']} ({result['confidence']:.3f})")
        
        return results


# Initialize AST detector
ast_detector = TrojanDetectorAST()
print("✅ AST Detector ready!")

# CELL 7: Complete Detection Pipeline

Combines DFG + AST results
Ensemble prediction (voting)
Calculates metrics (precision, recall, F1)
Saves results to JSON

In [ ]:
class CompleteTrojanDetectionPipeline:
    """Complete pipeline for trojan detection with both DFG and AST"""
    
    def __init__(self):
        self.dfg_detector = TrojanDetectorDFG()
        self.ast_detector = TrojanDetectorAST()
        
    def test_single_file(self, verilog_file):
        """
        Test a single Verilog file with both DFG and AST models
        
        Args:
            verilog_file: Path to .v or .sv file
            
        Returns:
            dict with combined results
        """
        print(f"\n{'='*60}")
        print(f"🔬 COMPLETE TROJAN DETECTION - SINGLE FILE")
        print(f"{'='*60}")
        print(f"📄 File: {verilog_file}")
        
        # Test with DFG
        dfg_result = self.dfg_detector.test_single_design(verilog_file)
        
        # Test with AST
        ast_result = self.ast_detector.test_single_design(verilog_file)
        
        # Combined result
        combined = {
            "file": str(verilog_file),
            "timestamp": datetime.now().isoformat(),
            "dfg_result": dfg_result,
            "ast_result": ast_result,
            "ensemble_prediction": self._ensemble_prediction(dfg_result, ast_result)
        }
        
        print(f"\n📊 FINAL RESULTS:")
        print(f"   DFG:      {dfg_result['prediction']} ({dfg_result['confidence']:.3f})")
        print(f"   AST:      {ast_result['prediction']} ({ast_result['confidence']:.3f})")
        print(f"   Ensemble: {combined['ensemble_prediction']['prediction']} "
              f"({combined['ensemble_prediction']['confidence']:.3f})")
        
        return combined
    
    def test_full_dataset(self):
        """Test complete dataset with both models"""
        
        print(f"\n{'='*60}")
        print(f"🔬 COMPLETE TROJAN DETECTION - FULL DATASET")
        print(f"{'='*60}")
        
        # Generate graphs for both types
        print(f"\n1️⃣ Generating DFG graphs...")
        dfg_gen = GraphGenerator(graph_type="DFG")
        dfg_graphs = dfg_gen.generate_graphs_from_dataset()
        
        print(f"\n2️⃣ Generating AST graphs...")
        ast_gen = GraphGenerator(graph_type="AST")
        ast_graphs = ast_gen.generate_graphs_from_dataset()
        
        # Test with both models
        print(f"\n3️⃣ Testing with DFG model...")
        dfg_results = self.dfg_detector.test_dataset(dfg_graphs)
        
        print(f"\n4️⃣ Testing with AST model...")
        ast_results = self.ast_detector.test_dataset(ast_graphs)
        
        # Combine results
        combined_results = self._combine_results(dfg_results, ast_results)
        
        # Calculate metrics
        metrics = self._calculate_metrics(combined_results)
        
        # Print summary
        self._print_summary(metrics, combined_results)
        
        # Save results
        output_file = "/kaggle/working/trojan_detection_results.json"
        self._save_results(combined_results, metrics, output_file)
        
        return combined_results, metrics
    
    def _ensemble_prediction(self, dfg_result, ast_result):
        """Combine DFG and AST predictions"""
        
        # Average probabilities
        avg_prob_trojan = (dfg_result['prob_trojan'] + ast_result['prob_trojan']) / 2
        avg_prob_clean = (dfg_result['prob_clean'] + ast_result['prob_clean']) / 2
        
        # Vote-based prediction (if either detects trojan with high confidence)
        if dfg_result['prediction'] == "TROJAN" or ast_result['prediction'] == "TROJAN":
            prediction = "TROJAN"
            confidence = max(dfg_result['prob_trojan'], ast_result['prob_trojan'])
        else:
            prediction = "CLEAN"
            confidence = max(dfg_result['prob_clean'], ast_result['prob_clean'])
        
        return {
            "prediction": prediction,
            "confidence": confidence,
            "avg_prob_trojan": avg_prob_trojan,
            "avg_prob_clean": avg_prob_clean
        }
    
    def _combine_results(self, dfg_results, ast_results):
        """Combine DFG and AST results"""
        
        combined = []
        for dfg_res, ast_res in zip(dfg_results, ast_results):
            ensemble = self._ensemble_prediction(dfg_res, ast_res)
            
            combined.append({
                "design_name": dfg_res['design_name'],
                "design_type": dfg_res['design_type'],
                "dfg": dfg_res,
                "ast": ast_res,
                "ensemble": ensemble
            })
        
        return combined
    
    def _calculate_metrics(self, combined_results):
        """Calculate detection metrics"""
        
        metrics = {
            "dfg": {"tp": 0, "tn": 0, "fp": 0, "fn": 0},
            "ast": {"tp": 0, "tn": 0, "fp": 0, "fn": 0},
            "ensemble": {"tp": 0, "tn": 0, "fp": 0, "fn": 0}
        }
        
        for result in combined_results:
            is_trojan = result['design_type'] == "TjIn"
            
            for model in ['dfg', 'ast', 'ensemble']:
                pred_trojan = result[model]['prediction'] == "TROJAN"
                
                if is_trojan and pred_trojan:
                    metrics[model]['tp'] += 1
                elif not is_trojan and not pred_trojan:
                    metrics[model]['tn'] += 1
                elif not is_trojan and pred_trojan:
                    metrics[model]['fp'] += 1
                else:
                    metrics[model]['fn'] += 1
        
        # Calculate precision, recall, F1
        for model in metrics:
            tp = metrics[model]['tp']
            fp = metrics[model]['fp']
            fn = metrics[model]['fn']
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            metrics[model]['precision'] = precision
            metrics[model]['recall'] = recall
            metrics[model]['f1'] = f1
            metrics[model]['detection_rate'] = recall
        
        return metrics
    
    def _print_summary(self, metrics, results):
        """Print detection summary"""
        
        print(f"\n{'='*60}")
        print(f"📊 DETECTION SUMMARY")
        print(f"{'='*60}")
        
        for model in ['dfg', 'ast', 'ensemble']:
            model_name = model.upper()
            m = metrics[model]
            
            print(f"\n{model_name} Model:")
            print(f"   Detection Rate: {m['detection_rate']*100:.1f}%")
            print(f"   Precision:      {m['precision']*100:.1f}%")
            print(f"   Recall:         {m['recall']*100:.1f}%")
            print(f"   F1 Score:       {m['f1']*100:.1f}%")
            print(f"   True Positives: {m['tp']}")
            print(f"   False Negatives: {m['fn']}")
        
        print(f"\n{'='*60}")
    
    def _save_results(self, results, metrics, output_file):
        """Save results to JSON"""
        
        output = {
            "metadata": {
                "timestamp": datetime.now().isoformat(),
                "total_designs": len(results),
                "device": DEVICE
            },
            "metrics": metrics,
            "results": results
        }
        
        with open(output_file, 'w') as f:
            json.dump(output, f, indent=2)
        
        print(f"\n💾 Results saved to: {output_file}")


# Initialize complete pipeline
pipeline = CompleteTrojanDetectionPipeline()
print("✅ Complete Pipeline ready!")


# CELL 8: Quick Test Functions

Ready-to-use wrapper functions
Just call and get results

In [ ]:
def quick_test_single_file(verilog_file):
    """
    Quick function to test a single Verilog file
    
    Usage:
        result = quick_test_single_file("/path/to/design.v")
    """
    return pipeline.test_single_file(verilog_file)


def quick_test_full_dataset():
    """
    Quick function to test full dataset
    
    Usage:
        results, metrics = quick_test_full_dataset()
    """
    return pipeline.test_full_dataset()


print("\n" + "="*60)
print("🎉 SETUP COMPLETE!")
print("="*60)
print("\n📝 USAGE:")
print("\n1️⃣ Test a single Verilog file:")
print("   result = quick_test_single_file('/kaggle/working/trojan_output/RISCV_ORIGINAL.sv')")
print("\n2️⃣ Test full dataset (all trojan files):")
print("   results, metrics = quick_test_full_dataset()")
print("\n3️⃣ Test with DFG only:")
print("   dfg_result = dfg_detector.test_single_design('/path/to/file.v')")
print("\n4️⃣ Test with AST only:")
print("   ast_result = ast_detector.test_single_design('/path/to/file.v')")
print("="*60)